# Customer Segmentation using K-Means Clustering
**Course:** Data Mining & Data Warehousing Lab (18B1WCI675)  
**Institute:** Jaypee University of Information Technology, Waknaghat  
**Dataset:** Mall Customers Dataset  

---
### Objective
Group customers into distinct clusters based on purchasing behavior using the K-Means algorithm. Identify optimal K using the Elbow Method and Silhouette Score. Visualize clusters and derive business insights.

---

## Section 1 — Install & Import Libraries

In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

print('1. All libraries imported successfully.')

## Section 2 — Load Dataset

In [ ]:
# 2. Load Mall Customers dataset directly from URL
URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/mall_customers.csv'

df = pd.read_csv(URL)

# Standardize column names
df.columns = df.columns.str.strip().str.replace(' ', '_')
print('2. Dataset loaded.')
print(f'   Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'   Columns: {list(df.columns)}')

## Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# 3a. Basic info
print('3a. Dataset Info:')
print('-' * 40)
df.info()
print()
print('3b. First 5 rows:')
df.head()

In [ ]:
# 3c. Statistical summary
print('3c. Statistical Summary:')
print('-' * 40)
print(df.describe().round(2))

In [ ]:
# 3d. Null check
print('3d. Null Values per Column:')
print('-' * 40)
nulls = df.isnull().sum()
print(nulls)
print(f'\n   Total nulls: {nulls.sum()}')

In [ ]:
# 3e. EDA Visualizations
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDA — Feature Distributions', fontsize=14, fontweight='bold', y=1.02)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

colors = ['#00d4ff', '#7c3aed', '#10b981']
for i, col in enumerate(numeric_cols[:3]):
    axes[i].hist(df[col], bins=20, color=colors[i], edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' '), fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('3e. EDA distribution plots saved.')

In [ ]:
# 3f. Correlation heatmap
plt.figure(figsize=(6, 4))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('3f. Correlation heatmap saved.')

In [ ]:
# 3g. Gender distribution (if present)
if 'Genre' in df.columns or 'Gender' in df.columns:
    gcol = 'Genre' if 'Genre' in df.columns else 'Gender'
    plt.figure(figsize=(5, 4))
    df[gcol].value_counts().plot(kind='bar', color=['#00d4ff', '#ef4444'],
                                  edgecolor='white', alpha=0.9)
    plt.title('Gender Distribution', fontweight='bold')
    plt.xlabel('Gender')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('eda_gender.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('3g. Gender distribution plot saved.')

## Section 4 — Preprocessing

In [ ]:
# 4a. Select features for clustering
# Using Annual Income and Spending Score — most discriminative for customer segmentation
FEATURE_COLS = [c for c in numeric_cols if c.lower() not in ['customerid', 'customer_id', 'id']]

print('4a. Numeric features available for clustering:')
for i, c in enumerate(FEATURE_COLS):
    print(f'   [{i}] {c}')

# Default: use last two (typically Income + Spending Score)
SELECTED = FEATURE_COLS[-2:]
print(f'\n   Selected for clustering: {SELECTED}')

In [ ]:
# 4b. Extract feature matrix
X = df[SELECTED].values

# 4c. Handle missing values
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# 4d. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('4b. Feature matrix extracted.')
print(f'   X shape: {X.shape}')
print(f'   X_scaled mean (should be ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'   X_scaled std  (should be ~1): {X_scaled.std(axis=0).round(4)}')

## Section 5 — Optimal K Selection (Elbow + Silhouette)

In [ ]:
# 5a. Compute WCSS and Silhouette Score for K = 2 to 10
K_RANGE = range(2, 11)
wcss = []
sil_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

print('5a. WCSS and Silhouette Scores computed.')
print(f'\n{"K":>4} {"WCSS":>10} {"Silhouette":>12}')
print('-' * 30)
for k, w, s in zip(K_RANGE, wcss, sil_scores):
    print(f'{k:>4} {w:>10.2f} {s:>12.4f}')

In [ ]:
# 5b. Plot Elbow Curve + Silhouette Score
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Elbow
ax1.plot(list(K_RANGE), wcss, 'o-', color='#00d4ff', linewidth=2.5, markersize=7)
ax1.fill_between(list(K_RANGE), wcss, alpha=0.08, color='#00d4ff')
ax1.set_title('Elbow Method — WCSS vs K', fontweight='bold', fontsize=12)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Within-Cluster Sum of Squares')
ax1.grid(alpha=0.3)
ax1.set_xticks(list(K_RANGE))

# Silhouette
ax2.plot(list(K_RANGE), sil_scores, 's-', color='#10b981', linewidth=2.5, markersize=7)
ax2.fill_between(list(K_RANGE), sil_scores, alpha=0.08, color='#10b981')
best_k_sil = list(K_RANGE)[sil_scores.index(max(sil_scores))]
ax2.axvline(x=best_k_sil, color='#ef4444', linestyle='--', alpha=0.7, label=f'Best K={best_k_sil}')
ax2.set_title('Silhouette Score vs K', fontweight='bold', fontsize=12)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xticks(list(K_RANGE))

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'5b. Best K by Silhouette Score: {best_k_sil}')

In [ ]:
# 5c. Set optimal K
OPTIMAL_K = 5   # Change this based on elbow + silhouette analysis
print(f'5c. Optimal K selected: {OPTIMAL_K}')
print('    (Adjust OPTIMAL_K above if your elbow/silhouette suggests a different value)')

## Section 6 — Train Final K-Means Model

In [ ]:
# 6a. Fit final model
kmeans_final = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=10, random_state=42)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'6a. K-Means model fitted with K={OPTIMAL_K}.')
print(f'    Final inertia (WCSS): {kmeans_final.inertia_:.2f}')
print(f'    Silhouette Score: {silhouette_score(X_scaled, df["Cluster"]):.4f}')
print()
print('6b. Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# 6c. Label clusters semantically based on centroid analysis
centers_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=SELECTED)
centers_df['Cluster'] = range(OPTIMAL_K)

print('6c. Cluster Centroids (original scale):')
print('-' * 50)
print(centers_df.round(2).to_string(index=False))

# Auto-generate semantic labels based on feature ranks
f1, f2 = SELECTED[0], SELECTED[1]
labels_map = {}
for i, row in centers_df.iterrows():
    v1 = 'High' if row[f1] > centers_df[f1].median() else 'Low'
    v2 = 'High' if row[f2] > centers_df[f2].median() else 'Low'
    labels_map[i] = f'{v1} {f1.replace("_"," ")} / {v2} {f2.replace("_"," ")}'

df['Segment'] = df['Cluster'].map(labels_map)
print('\n6d. Semantic Segment Labels:')
for k, v in labels_map.items():
    print(f'   Cluster {k}: {v}')

## Section 7 — Visualizations

In [ ]:
# 7a. 2D Cluster Scatter Plot
PALETTE = ['#00d4ff', '#7c3aed', '#10b981', '#f59e0b', '#ef4444',
           '#ec4899', '#06b6d4', '#84cc16']

plt.figure(figsize=(9, 6))

for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    plt.scatter(
        df.loc[mask, SELECTED[0]],
        df.loc[mask, SELECTED[1]],
        s=60, alpha=0.8,
        color=PALETTE[cluster_id % len(PALETTE)],
        label=f'Cluster {cluster_id}: {labels_map[cluster_id]}'
    )

# Plot centroids
plt.scatter(
    centers_df[SELECTED[0]], centers_df[SELECTED[1]],
    s=250, marker='X', c='white', edgecolors='black', linewidths=1.5,
    zorder=5, label='Centroids'
)

plt.title(f'Customer Segments — K-Means (K={OPTIMAL_K})', fontweight='bold', fontsize=13)
plt.xlabel(SELECTED[0].replace('_', ' '))
plt.ylabel(SELECTED[1].replace('_', ' '))
plt.legend(loc='upper left', fontsize=8, framealpha=0.9)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('cluster_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('7a. Cluster scatter plot saved.')

In [ ]:
# 7b. Cluster size bar chart
cluster_counts = df['Cluster'].value_counts().sort_index()

plt.figure(figsize=(7, 4))
bars = plt.bar(
    [f'C{i}' for i in cluster_counts.index],
    cluster_counts.values,
    color=[PALETTE[i % len(PALETTE)] for i in cluster_counts.index],
    edgecolor='white', alpha=0.9
)
for bar, val in zip(bars, cluster_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.title('Cluster Size Distribution', fontweight='bold', fontsize=12)
plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print('7b. Cluster size chart saved.')

In [ ]:
# 7c. Feature distribution per cluster (boxplots)
fig, axes = plt.subplots(1, len(SELECTED), figsize=(6 * len(SELECTED), 5))
if len(SELECTED) == 1:
    axes = [axes]

for ax, feat in zip(axes, SELECTED):
    cluster_data = [df[df['Cluster'] == c][feat].values for c in sorted(df['Cluster'].unique())]
    bp = ax.boxplot(cluster_data, patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_title(f'{feat.replace("_"," ")} by Cluster', fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_xticklabels([f'C{i}' for i in sorted(df['Cluster'].unique())])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('7c. Boxplots saved.')

In [ ]:
# 7d. PCA 2D projection (for datasets with more than 2 features)
if len(SELECTED) > 2:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    explained = pca.explained_variance_ratio_.sum() * 100

    plt.figure(figsize=(8, 6))
    for cluster_id in sorted(df['Cluster'].unique()):
        mask = df['Cluster'] == cluster_id
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=60, alpha=0.8, color=PALETTE[cluster_id % len(PALETTE)],
                    label=f'Cluster {cluster_id}')

    plt.title(f'PCA 2D Projection ({explained:.1f}% variance explained)', fontweight='bold')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('cluster_pca.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'7d. PCA projection saved. ({explained:.1f}% variance explained)')
else:
    print('7d. PCA skipped (only 2 features — scatter already in 2D).')

## Section 8 — Cluster Profile & Business Insights

In [ ]:
# 8a. Cluster profile summary
profile = df.groupby('Cluster')[SELECTED].agg(['mean', 'min', 'max', 'count'])
print('8a. Cluster Profile Summary:')
print('=' * 60)
print(profile.round(2))

In [ ]:
# 8b. Business Insight report
print('8b. Business Insights & Marketing Recommendations')
print('=' * 60)
for cluster_id in sorted(df['Cluster'].unique()):
    grp = df[df['Cluster'] == cluster_id]
    size = len(grp)
    pct = size / len(df) * 100
    print(f'\nCluster {cluster_id} — {labels_map[cluster_id]}')
    print(f'  Size   : {size} customers ({pct:.1f}%)')
    for feat in SELECTED:
        print(f'  Avg {feat.replace("_"," "):25s}: {grp[feat].mean():.2f}')
    print(f'  Segment: {df[df["Cluster"]==cluster_id]["Segment"].iloc[0]}')

## Section 9 — Export Results

In [ ]:
# 9. Save labeled dataset
output_path = 'customer_segments_output.csv'
df.to_csv(output_path, index=False)
print(f'9. Labeled dataset saved to: {output_path}')
print(f'   Columns: {list(df.columns)}')
print(f'   Shape  : {df.shape}')
print()
print('All outputs saved:')
print('  - eda_distributions.png')
print('  - eda_correlation.png')
print('  - elbow_silhouette.png')
print('  - cluster_scatter.png')
print('  - cluster_sizes.png')
print('  - cluster_boxplots.png')
print('  - customer_segments_output.csv')